In [ ]:
# Diagnostic: confirm what is actually mounted under /kaggle/input before
# relying on any specific dataset path (added after the first run failed
# with pip reporting the wheelhouse path as non-existing; the second run
# revealed /kaggle/input only has a single 'datasets' entry, so datasets are
# nested under it rather than mounted directly at /kaggle/input/<slug>).
import os
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
    print(f"{'  ' * depth}{root} -> dirs={dirs} files={files[:5]}{'...' if len(files) > 5 else ''}")
    if depth >= 3:
        dirs[:] = []

In [ ]:
# ADR 0048 validation notebook: confirm OLMo-2-1124-7B (pre-quantized 4bit)
# plus Unsloth/bitsandbytes/peft/trl/xformers load and run one inference,
# entirely offline (enable_internet=false in kernel-metadata.json).
import subprocess
import os

# Resolve the actual mounted paths by walking /kaggle/input, since the
# diagnostic cell above showed datasets are nested (e.g. under a 'datasets'
# subfolder) rather than mounted directly at /kaggle/input/<slug>.
_input_root = "/kaggle/input"

def _resolve(keyword):
    for root, dirs, _files in os.walk(_input_root):
        for d in dirs:
            if keyword in d:
                return os.path.join(root, d)
    raise FileNotFoundError(f"no mounted dataset dir containing '{keyword}' under {_input_root}")

WHEELHOUSE = _resolve("wheelhouse")
MODEL_DIR = _resolve("olmo2")
print("WHEELHOUSE resolved to:", WHEELHOUSE)
print("MODEL_DIR resolved to:", MODEL_DIR)

cmd = [
    "pip", "install", "--no-index", "--no-deps",
    "--find-links", WHEELHOUSE,
    "unsloth", "unsloth_zoo", "bitsandbytes", "trl", "xformers",
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-3000:])
print(result.stderr[-3000:])
print("pip install returncode:", result.returncode)
assert result.returncode == 0, "offline pip install failed"

In [ ]:
# Confirm internet is really disabled before touching the model
import socket
socket.setdefaulttimeout(5)
try:
    import urllib.request
    urllib.request.urlopen("https://huggingface.co")
    print("INTERNET IS REACHABLE (unexpected)")
except Exception as exc:
    print(f"internet unreachable as expected: {type(exc).__name__}: {exc}")

In [ ]:
import importlib
for pkg in ["unsloth", "unsloth_zoo", "bitsandbytes", "trl", "xformers"]:
    mod = importlib.import_module(pkg)
    print(f"{pkg}: OK, version={getattr(mod, '__version__', 'unknown')}")

In [ ]:
# Load the pre-quantized OLMo-2-1124-7B checkpoint directly from the attached
# dataset path, no HuggingFace Hub access, no HF_HUB_OFFLINE needed since we
# pass a local directory path directly.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_DIR,
    max_seq_length=2048,
    load_in_4bit=True,
)
print("Model loaded OK from", MODEL_DIR)

In [ ]:
FastLanguageModel.for_inference(model)
prompt = "Input:\n0 0\n0 0\n\nOutput:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output_ids = model.generate(**inputs, max_new_tokens=16)
text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("GENERATION OK")
print(repr(text))